In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

In [7]:
#load_longbench_dataset_with_chunking()

# load_longbench_with_refined_chunking()


#chunks = split_chunks_with_cleaning(file_path)

#print(f"清洗并分块完成，共提取出 {len(chunks)} 条高质量文档块。")

# 预览一下清洗后的效果
# for i, chunk in enumerate(chunks[:3], 1):
#     print(f"--- 经清洗后的块 {i} ---")
#     print(f"{chunk[:200]}") # 看看是否还有 <br>
#     print()

#update_eval_file('longbench_eval.json', 'longbench_eval_with_gt.json')

# --- 执行示例 ---
# 1. 运行你之前的 generate_contextual_chunks 得到列表
#enhanced_chunks = generate_contextual_chunks('longbench_kb.md')

# 2. 保存为原样格式的文件
#save_enhanced_kb_to_disk(enhanced_chunks, "longbench_kb_contextual.md")

#save_embeddings(real_ids, chunks, embeddings)
#print(real_ids[0],embeddings[0])
#client.delete_collection("baai_bge_collection")
#client.delete_collection("contextual_collection")

🚀 正在启动：智能清洗 + 递归分块模式...
✅ 构建完成！知识库块数: 481，测试题数: 50
📊 样例：第一题被切分为 14 个语义连贯的知识块。


In [10]:

#real_ids, chunks = parse_kb_file('dureader_kb.md')
#embeddings = embedding_model.encode(chunks, normalize_embeddings=True).tolist()
#save_embeddings(real_ids, chunks, embeddings)

成功存入 1263 条向量数据。


In [12]:
from rag.src.rerank_utils2 import adaptive_evaluate_logic, mmr_listwise_rerank


In [1]:
import sys
print(sys.executable)

/data1/yesiliang/rag/.venv/bin/python


In [8]:
from rag.src.generate import evaluate_rag_system, strategy_pairwise, strategy_listwise, strategy_colbert, strategy_pointwise, strategy_baseline
from rag.src.generate import strategy_colbert_listwise, strategy_setselection
import json

# 1. 加载数据
with open("/data1/yesiliang/rag/data/dureader/dureader_eval.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)
test_questions = test_data["test"]

# 2. 【核心修改点】在这里选择你想使用的策略函数名
CURRENT_STRATEGY = strategy_baseline
#CURRENT_STRATEGY = strategy_pointwise
#CURRENT_STRATEGY = strategy_listwise
#CURRENT_STRATEGY = strategy_colbert 
#CURRENT_STRATEGY = strategy_colbert_listwise 
#CURRENT_STRATEGY = strategy_setselection

# 3. 设置对应的输出文件名
output_mapping = {
    strategy_baseline: "dureader_baseline.json",
    strategy_pointwise: "bge_pointwise_contextual.json",
    strategy_colbert: "bge_colbert.json",
    strategy_listwise: "deepseek_listwise_contextual.json",
    strategy_pairwise: "deepseek_pairwise.json",
    strategy_colbert_listwise: "bge_colbert_deepseek_listwise.json",
    strategy_setselection:"dureader_setselection.json"
}
output_file = output_mapping[CURRENT_STRATEGY]
output_path = "/data1/yesiliang/rag/results"/output_file
# 4. 执行并保存
test_results = evaluate_rag_system(test_questions, CURRENT_STRATEGY, top_k_for_gen=5, skip_retrieval=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(test_results, f, indent=2, ensure_ascii=False)

print(f"评估完成！使用策略: {CURRENT_STRATEGY.__name__}, 结果存至: {output_path}")

开始评估，策略名称: strategy_setselection，总计 50 个问题...


100%|███████████████████████████████████████████████████████████████████████████████| 50/50 [53:05<00:00, 63.70s/it]

评估完成！使用策略: strategy_setselection, 结果存至: dureader_setselection.json


In [4]:
from rag.src.generate import calculate_metrics

GT_FILE = 'longbench_eval.json'
PRED_FILE = 'bge_pointwise_contextual.json' # 你的检索结果文件名
    
calculate_metrics(GT_FILE, PRED_FILE)


检索效果评估报告
----------------------------------------
评估题目总数:  50
Hit Rate @ 1:  0.3600
Hit Rate @ 2:  0.6400
Hit Rate @ 20: 0.9600
MRR:           0.5678


In [9]:
from rag.src.generate import run_gen_evaluation

run_gen_evaluation("dureader_setselection.json")


生成质量评估报告: dureader_setselection.json
---------------------------------------------
评估样本数: 50
Exact Match (EM): 0.0400
F1 Score:         0.3296
Precision (P):    0.4834
Recall (R):       0.3650
